# 09 Statistical validation

This notebook uses the matched 823-patient split and the model selections from notebook 08.

It adds:
- test class counts and error counts
- 95% bootstrap confidence intervals
- exact McNemar tests
- paired bootstrap differences for accuracy and AUC

This notebook does not create new train/test splits.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from scipy.stats import binomtest

from sklearn.base import clone
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

import os

BASE_DIR = Path(
    os.environ.get("FUSION_REPO_DIR", str(Path.cwd()))
).expanduser().resolve()

if BASE_DIR.name in {"Preprocessing", "Training"}:
    BASE_DIR = BASE_DIR.parent

if not (BASE_DIR / "Training" / "07_controlled_early_late_fusion.ipynb").is_file():
    raise RuntimeError(
        "Start Jupyter from the repository directory or set FUSION_REPO_DIR."
    )
DATA_DIR = Path(
    os.environ.get("FUSION_DATA_DIR", str(BASE_DIR / "Data"))
).expanduser().resolve()
ANALYSIS_DIR = BASE_DIR
PREP_DIR = ANALYSIS_DIR / "Preprocessing"
CONTROL_DIR = ANALYSIS_DIR / "Controlled-Fusion"
FEATURE_DIR = CONTROL_DIR / "Features"
MATCHED_DIR = ANALYSIS_DIR / "Matched-823-Fusion"
RESULT_DIR = MATCHED_DIR / "Results"
OUT_DIR = MATCHED_DIR / "Statistical-Validation"

OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS_PATH = DATA_DIR / "multiomics_labels.tsv"
SPLIT_PATH = DATA_DIR / "train_test_split.csv"

CLINICAL_ORIGINAL_PATH = PREP_DIR / "clinical_features.csv"
RNA_ORIGINAL_PATH = PREP_DIR / "rna_selected_features.csv"
METH_ORIGINAL_PATH = PREP_DIR / "methylation_selected_features.csv"
WSI_ORIGINAL_PATH = FEATURE_DIR / "wsi_features_split_safe_100.csv"

CLINICAL_EQUAL40_PATH = FEATURE_DIR / "clinical_features_fusion40.csv"
RNA_EQUAL40_PATH = FEATURE_DIR / "rna_features_fusion40.csv"
METH_EQUAL40_PATH = FEATURE_DIR / "methylation_features_fusion40.csv"
WSI_EQUAL40_PATH = FEATURE_DIR / "wsi_features_split_safe_40.csv"

BEST_EARLY_PATH = RESULT_DIR / "best_early_matched_823.csv"
BEST_LATE_PATH = RESULT_DIR / "best_late_matched_823.csv"

RANDOM_STATE = 42
N_BOOTSTRAP = 5000

print("Output directory:", OUT_DIR)


In [ ]:
def normalize_id(x):
    return str(x).strip().upper()[:12]


def load_feature_csv(path):
    df = pd.read_csv(path, index_col=0)
    df.index = [normalize_id(x) for x in df.index]
    df = df[~df.index.duplicated(keep="first")]
    df = df.apply(pd.to_numeric, errors="coerce")
    if df.isna().any().any():
        raise ValueError(f"Missing values found in {path}")
    return df


labels_df = pd.read_csv(LABELS_PATH, sep="\t")
labels_df["patient_id"] = labels_df["patient_id"].map(normalize_id)

labels = (
    labels_df
    .drop_duplicates("patient_id")
    .set_index("patient_id")["subtype_simple"]
    .astype(str)
    .str.strip()
    .map({"LUAD": 0, "LUSC": 1})
    .dropna()
    .astype(int)
)

split_df = pd.read_csv(SPLIT_PATH)
if "patient_id" not in split_df.columns:
    split_df = pd.read_csv(SPLIT_PATH, index_col=0).reset_index()
    split_df = split_df.rename(columns={split_df.columns[0]: "patient_id"})

split_df["patient_id"] = split_df["patient_id"].map(normalize_id)
split_df = split_df.drop_duplicates("patient_id").set_index("patient_id")
split_df["split"] = split_df["split"].astype(str).str.lower().str.strip()

clinical_original = load_feature_csv(CLINICAL_ORIGINAL_PATH)
rna_original = load_feature_csv(RNA_ORIGINAL_PATH)
meth_original = load_feature_csv(METH_ORIGINAL_PATH)
wsi_original = load_feature_csv(WSI_ORIGINAL_PATH)

clinical_equal40 = load_feature_csv(CLINICAL_EQUAL40_PATH)
rna_equal40 = load_feature_csv(RNA_EQUAL40_PATH)
meth_equal40 = load_feature_csv(METH_EQUAL40_PATH)
wsi_equal40 = load_feature_csv(WSI_EQUAL40_PATH)

master_set = (
    set(labels.index)
    & set(split_df.index)
    & set(clinical_original.index)
    & set(rna_original.index)
    & set(meth_original.index)
    & set(wsi_original.index)
    & set(clinical_equal40.index)
    & set(rna_equal40.index)
    & set(meth_equal40.index)
    & set(wsi_equal40.index)
)

MASTER_IDS = [pid for pid in split_df.index if pid in master_set]
TRAIN_IDS = [pid for pid in MASTER_IDS if split_df.loc[pid, "split"] == "train"]
TEST_IDS = [pid for pid in MASTER_IDS if split_df.loc[pid, "split"] == "test"]

if len(MASTER_IDS) != 823:
    raise RuntimeError(f"Expected 823 patients, found {len(MASTER_IDS)}")

ORIGINAL = {
    "Clinical": clinical_original.loc[MASTER_IDS].copy(),
    "RNA-Seq": rna_original.loc[MASTER_IDS].copy(),
    "Methylation": meth_original.loc[MASTER_IDS].copy(),
    "WSI": wsi_original.loc[MASTER_IDS].copy(),
}

EQUAL40 = {
    "Clinical": clinical_equal40.loc[MASTER_IDS].copy(),
    "RNA-Seq": rna_equal40.loc[MASTER_IDS].copy(),
    "Methylation": meth_equal40.loc[MASTER_IDS].copy(),
    "WSI": wsi_equal40.loc[MASTER_IDS].copy(),
}

FEATURE_SETS = {
    "Original": ORIGINAL,
    "Equal40": EQUAL40,
}

Y_TRAIN = labels.loc[TRAIN_IDS].values
Y_TEST = labels.loc[TEST_IDS].values

print("Total:", len(MASTER_IDS))
print("Train:", len(TRAIN_IDS))
print("Test:", len(TEST_IDS))
print("Train LUAD:", int((Y_TRAIN == 0).sum()))
print("Train LUSC:", int((Y_TRAIN == 1).sum()))
print("Test LUAD:", int((Y_TEST == 0).sum()))
print("Test LUSC:", int((Y_TEST == 1).sum()))


In [ ]:
CLASSIFIERS = {
    "LDA": LinearDiscriminantAnalysis(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "CART": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "NB": GaussianNB(),
    "SVM": SVC(
        kernel="rbf",
        C=1.0,
        probability=True,
        random_state=RANDOM_STATE,
    ),
    "LR": LogisticRegression(
        solver="saga",
        C=1.0,
        max_iter=2000,
        random_state=RANDOM_STATE,
    ),
}

COMBINATIONS = {
    "Unimodal-RNA-Seq": ["RNA-Seq"],
    "Omics": ["RNA-Seq", "Methylation"],
    "Clinical+Omics": ["Clinical", "RNA-Seq", "Methylation"],
    "Full": ["Clinical", "RNA-Seq", "Methylation", "WSI"],
}

best_early = pd.read_csv(BEST_EARLY_PATH)
best_late = pd.read_csv(BEST_LATE_PATH)

print("Loaded model selections from notebook 08.")


In [ ]:
def make_model(classifier_name):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", clone(CLASSIFIERS[classifier_name])),
    ])


def fuse_probabilities(probability_list, rule):
    probs = np.column_stack(probability_list)
    probs = np.clip(probs, 1e-8, 1 - 1e-8)

    if rule == "Maximum":
        return probs.max(axis=1)
    if rule == "Sum":
        return probs.mean(axis=1)
    if rule == "Product":
        return probs.prod(axis=1)

    raise ValueError(f"Unknown fusion rule: {rule}")


def build_early_matrix(modality_dict, modalities):
    blocks = []
    for modality in modalities:
        block = modality_dict[modality].loc[MASTER_IDS].copy()
        block.columns = [f"{modality}__{c}" for c in block.columns]
        blocks.append(block)
    return pd.concat(blocks, axis=1)


def get_selected_row(source_df, budget, combination):
    row = source_df[
        (source_df["budget"] == budget)
        & (source_df["combination"] == combination)
    ]
    if len(row) != 1:
        raise ValueError(
            f"Expected one selected row for {budget}, {combination}; found {len(row)}"
        )
    return row.iloc[0]


def compute_metrics(y_true, y_prob, threshold):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "auc_roc": roc_auc_score(y_true, y_prob),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "n_errors": int((y_pred != y_true).sum()),
    }


def fit_selected_early(budget, combination):
    modality_dict = FEATURE_SETS[budget]
    modalities = COMBINATIONS[combination]
    row = get_selected_row(best_early, budget, combination)

    X = build_early_matrix(modality_dict, modalities)
    X_train = X.loc[TRAIN_IDS].values
    X_test = X.loc[TEST_IDS].values

    model = make_model(row["classifier"])
    model.fit(X_train, Y_TRAIN)

    test_prob = model.predict_proba(X_test)[:, 1]
    threshold = float(row["threshold"])

    return row, test_prob, threshold


def fit_selected_late(budget, combination):
    modality_dict = FEATURE_SETS[budget]
    modalities = COMBINATIONS[combination]
    row = get_selected_row(best_late, budget, combination)

    test_probabilities = []

    for modality in modalities:
        X = modality_dict[modality]
        X_train = X.loc[TRAIN_IDS].values
        X_test = X.loc[TEST_IDS].values

        model = make_model(row["classifier"])
        model.fit(X_train, Y_TRAIN)
        test_probabilities.append(model.predict_proba(X_test)[:, 1])

    test_prob = fuse_probabilities(
        test_probabilities,
        row["fusion_rule"],
    )
    threshold = float(row["threshold"])

    return row, test_prob, threshold


In [ ]:
prediction_frames = []
metric_rows = []

for budget in ["Original", "Equal40"]:
    selected = [
        ("Early", "Unimodal-RNA-Seq"),
        ("Early", "Omics"),
        ("Late", "Omics"),
        ("Early", "Clinical+Omics"),
        ("Late", "Clinical+Omics"),
        ("Early", "Full"),
        ("Late", "Full"),
    ]

    for strategy, combination in selected:
        if strategy == "Early":
            row, test_prob, threshold = fit_selected_early(budget, combination)
        else:
            row, test_prob, threshold = fit_selected_late(budget, combination)

        test_pred = (test_prob >= threshold).astype(int)
        metrics = compute_metrics(Y_TEST, test_prob, threshold)

        expected_acc = float(row["test_accuracy"])
        expected_auc = float(row["test_auc_roc"])

        if not np.isclose(metrics["accuracy"], expected_acc, atol=1e-10):
            raise RuntimeError(
                f"Accuracy mismatch for {budget}, {strategy}, {combination}"
            )

        if not np.isclose(metrics["auc_roc"], expected_auc, atol=1e-8):
            raise RuntimeError(
                f"AUC mismatch for {budget}, {strategy}, {combination}"
            )

        model_name = f"{budget}_{strategy}_{combination}"

        prediction_frames.append(pd.DataFrame({
            "patient_id": TEST_IDS,
            "y_true": Y_TEST,
            "y_prob": test_prob,
            "y_pred": test_pred,
            "budget": budget,
            "strategy": strategy,
            "combination": combination,
            "model_name": model_name,
        }))

        metric_rows.append({
            "budget": budget,
            "strategy": strategy,
            "combination": combination,
            "classifier": row["classifier"],
            "fusion_rule": row.get("fusion_rule", ""),
            "threshold": threshold,
            "n_test": len(Y_TEST),
            **metrics,
        })

predictions = pd.concat(prediction_frames, ignore_index=True)
metrics_df = pd.DataFrame(metric_rows)

predictions.to_csv(OUT_DIR / "selected_test_predictions.csv", index=False)
metrics_df.to_csv(OUT_DIR / "selected_test_metrics.csv", index=False)

display(metrics_df)


In [ ]:
def bootstrap_ci(y_true, y_prob, threshold, n_bootstrap=N_BOOTSTRAP, seed=42):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    idx0 = np.where(y_true == 0)[0]
    idx1 = np.where(y_true == 1)[0]

    rng = np.random.default_rng(seed)

    values = {
        "accuracy": [],
        "f1_macro": [],
        "auc_roc": [],
        "sensitivity": [],
        "specificity": [],
    }

    for _ in range(n_bootstrap):
        sample0 = rng.choice(idx0, size=len(idx0), replace=True)
        sample1 = rng.choice(idx1, size=len(idx1), replace=True)
        idx = np.concatenate([sample0, sample1])

        m = compute_metrics(
            y_true[idx],
            y_prob[idx],
            threshold,
        )

        for metric in values:
            values[metric].append(m[metric])

    rows = []

    point = compute_metrics(y_true, y_prob, threshold)

    for metric, vals in values.items():
        vals = np.asarray(vals, dtype=float)
        rows.append({
            "metric": metric,
            "estimate": point[metric],
            "ci_low": np.percentile(vals, 2.5),
            "ci_high": np.percentile(vals, 97.5),
        })

    return pd.DataFrame(rows)


ci_frames = []

for i, row in metrics_df.reset_index(drop=True).iterrows():
    block = predictions[predictions["model_name"] == (
        f"{row['budget']}_{row['strategy']}_{row['combination']}"
    )]

    ci = bootstrap_ci(
        block["y_true"].values,
        block["y_prob"].values,
        float(row["threshold"]),
        seed=RANDOM_STATE + i,
    )

    ci.insert(0, "combination", row["combination"])
    ci.insert(0, "strategy", row["strategy"])
    ci.insert(0, "budget", row["budget"])

    ci_frames.append(ci)

bootstrap_results = pd.concat(ci_frames, ignore_index=True)
bootstrap_results.to_csv(
    OUT_DIR / "bootstrap_confidence_intervals.csv",
    index=False,
)

display(bootstrap_results)


In [ ]:
def get_prediction_block(budget, strategy, combination):
    return (
        predictions[
            (predictions["budget"] == budget)
            & (predictions["strategy"] == strategy)
            & (predictions["combination"] == combination)
        ]
        .set_index("patient_id")
        .loc[TEST_IDS]
    )


def exact_mcnemar(block_a, block_b):
    y = block_a["y_true"].values
    pred_a = block_a["y_pred"].values
    pred_b = block_b["y_pred"].values

    correct_a = pred_a == y
    correct_b = pred_b == y

    a_only = int(np.sum(correct_a & ~correct_b))
    b_only = int(np.sum(~correct_a & correct_b))
    discordant = a_only + b_only

    if discordant == 0:
        p_value = 1.0
    else:
        p_value = binomtest(
            a_only,
            n=discordant,
            p=0.5,
            alternative="two-sided",
        ).pvalue

    return a_only, b_only, discordant, p_value


comparison_specs = [
    ("Omics: Early vs Late", ("Early", "Omics"), ("Late", "Omics")),
    (
        "Clinical+Omics: Early vs Late",
        ("Early", "Clinical+Omics"),
        ("Late", "Clinical+Omics"),
    ),
    ("Full: Early vs Late", ("Early", "Full"), ("Late", "Full")),
    (
        "RNA vs Full Early",
        ("Early", "Unimodal-RNA-Seq"),
        ("Early", "Full"),
    ),
    (
        "RNA vs Full Late",
        ("Early", "Unimodal-RNA-Seq"),
        ("Late", "Full"),
    ),
]

mcnemar_rows = []

for budget in ["Original", "Equal40"]:
    for name, a_spec, b_spec in comparison_specs:
        block_a = get_prediction_block(budget, *a_spec)
        block_b = get_prediction_block(budget, *b_spec)

        a_only, b_only, discordant, p_value = exact_mcnemar(
            block_a,
            block_b,
        )

        mcnemar_rows.append({
            "budget": budget,
            "comparison": name,
            "a_correct_b_wrong": a_only,
            "a_wrong_b_correct": b_only,
            "discordant_pairs": discordant,
            "p_value": p_value,
        })

mcnemar_df = pd.DataFrame(mcnemar_rows)
mcnemar_df.to_csv(OUT_DIR / "mcnemar_tests.csv", index=False)

display(mcnemar_df)


In [ ]:
def paired_bootstrap_difference(
    block_a,
    block_b,
    n_bootstrap=N_BOOTSTRAP,
    seed=42,
):
    y = block_a["y_true"].values

    prob_a = block_a["y_prob"].values
    prob_b = block_b["y_prob"].values

    pred_a = block_a["y_pred"].values
    pred_b = block_b["y_pred"].values

    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]

    rng = np.random.default_rng(seed)

    acc_diff = []
    auc_diff = []

    for _ in range(n_bootstrap):
        sample0 = rng.choice(idx0, size=len(idx0), replace=True)
        sample1 = rng.choice(idx1, size=len(idx1), replace=True)
        idx = np.concatenate([sample0, sample1])

        acc_a = accuracy_score(y[idx], pred_a[idx])
        acc_b = accuracy_score(y[idx], pred_b[idx])

        auc_a = roc_auc_score(y[idx], prob_a[idx])
        auc_b = roc_auc_score(y[idx], prob_b[idx])

        acc_diff.append(acc_b - acc_a)
        auc_diff.append(auc_b - auc_a)

    point_acc = accuracy_score(y, pred_b) - accuracy_score(y, pred_a)
    point_auc = roc_auc_score(y, prob_b) - roc_auc_score(y, prob_a)

    return {
        "accuracy_delta_b_minus_a": point_acc,
        "accuracy_ci_low": np.percentile(acc_diff, 2.5),
        "accuracy_ci_high": np.percentile(acc_diff, 97.5),
        "auc_delta_b_minus_a": point_auc,
        "auc_ci_low": np.percentile(auc_diff, 2.5),
        "auc_ci_high": np.percentile(auc_diff, 97.5),
    }


paired_rows = []
counter = 0

for budget in ["Original", "Equal40"]:
    for name, a_spec, b_spec in comparison_specs:
        block_a = get_prediction_block(budget, *a_spec)
        block_b = get_prediction_block(budget, *b_spec)

        result = paired_bootstrap_difference(
            block_a,
            block_b,
            seed=RANDOM_STATE + counter,
        )
        counter += 1

        paired_rows.append({
            "budget": budget,
            "comparison": name,
            **result,
        })

paired_df = pd.DataFrame(paired_rows)
paired_df.to_csv(
    OUT_DIR / "paired_bootstrap_differences.csv",
    index=False,
)

display(paired_df)


In [ ]:
print("Saved files:")
for path in sorted(OUT_DIR.glob("*.csv")):
    print(path.name)
